# DeepMatch 样例代码
- https://github.com/shenweichen/DeepMatch
- https://deepmatch.readthedocs.io/en/latest/

# 下载movielens-1M数据 安装依赖包

In [ ]:
! wget http://files.grouplens.org/datasets/movielens/ml-1m.zip -O ./ml-1m.zip 
! wget https://raw.githubusercontent.com/shenweichen/DeepMatch/master/examples/preprocess.py -O preprocess.py
! unzip -o ml-1m.zip 
! pip uninstall -y -q tensorflow
! pip install -q tensorflow-gpu==2.5.0
! pip install -q deepmatch

# 导入需要的库

In [1]:
import pandas as pd
from deepctr.feature_column import SparseFeat, VarLenSparseFeat
from preprocess import gen_data_set, gen_model_input
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model

from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss, NegativeSampler

# 这一行负责从 preprocess.py 文件中加载 gen_model_input 函数
from preprocess import gen_data_set, gen_model_input 

# 读取数据

In [2]:
data_path = "./"

unames = ['user_id','gender','age','occupation','zip']
user = pd.read_csv(data_path+'ml-1m/users.dat',sep='::',header=None,names=unames)
rnames = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(data_path+'ml-1m/ratings.dat',sep='::',header=None,names=rnames)
mnames = ['movie_id','title','genres']
movies = pd.read_csv(data_path+'ml-1m/movies.dat',sep='::',header=None,names=mnames,encoding="unicode_escape")
movies['genres'] = list(map(lambda x: x.split('|')[0], movies['genres'].values))

data = pd.merge(pd.merge(ratings,movies),user)#.iloc[:10000]

C:\Users\ZaoYangxinzi\AppData\Local\Temp\ipykernel_13896\1696079877.py:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  user = pd.read_csv(data_path+'ml-1m/users.dat',sep='::',header=None,names=unames)
C:\Users\ZaoYangxinzi\AppData\Local\Temp\ipykernel_13896\1696079877.py:6: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  ratings = pd.read_csv(data_path+'ml-1m/ratings.dat',sep='::',header=None,names=rnames)
C:\Users\ZaoYangxinzi\AppData\Local\Temp\ipykernel_13896\1696079877.py:8: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 cha

In [3]:
user

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


In [4]:
ratings

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [5]:
movies

,movie_id,title,genres
0,1,Toy Story (1995),Animation
1,2,Jumanji (1995),Adventure
2,3,Grumpier Old Men (1995),Comedy
3,4,Waiting to Exhale (1995),Comedy
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [6]:
data

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama,F,1,10,48067
1,1,661,3,978302109,James and the Giant Peach (1996),Animation,F,1,10,48067
2,1,914,3,978301968,My Fair Lady (1964),Musical,F,1,10,48067
3,1,3408,4,978300275,Erin Brockovich (2000),Drama,F,1,10,48067
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation,F,1,10,48067
...,...,...,...,...,...,...,...,...,...,...
1000204,4211,3791,2,965319075,Footloose (1984),Drama,M,45,5,77662
1000205,4211,3806,3,965319138,MacKenna's Gold (1969),Western,M,45,5,77662
1000206,4211,3840,4,965319197,Pumpkinhead (1988),Horror,M,45,5,77662
1000207,4211,3766,2,965319138,Missing in Action (1984),Action,M,45,5,77662


# 构建特征列，训练模型，导出embedding

In [3]:
#data = pd.read_csvdata = pd.read_csv("./movielens_sample.txt")
sparse_features = ["movie_id", "user_id",
                    "gender", "age", "occupation", "zip", "genres"]
SEQ_LEN = 50
negsample = 0

In [4]:
# 1.Label Encoding for sparse features,and process sequence features with `gen_date_set` and `gen_model_input`

feature_max_idx = {}
for feature in sparse_features:
    lbe = LabelEncoder()
    data[feature] = lbe.fit_transform(data[feature]) + 1
    feature_max_idx[feature] = data[feature].max() + 1  #记录每个特征的最大整数索引。这个数值决定了后续 Embedding 层（嵌入层） 的词表大小（Vocabulary Size）。


#构建去重画像 (Profile Extraction)，建立一个静态查找表（Lookup Table）
user_profile = data[["user_id", "gender", "age", "occupation", "zip"]].drop_duplicates('user_id')
item_profile = data[["movie_id"]].drop_duplicates('movie_id')
'''原始的 data 表是合并后的交互记录，同一个 User 对应多行数据。
这里提取出唯一的 User ID -> 用户属性 和 Movie ID -> 物品属性 的映射关系。'''
print('构建去重画像 (Profile Extraction)')
print(user_profile.shape)
print(user_profile.head())
print(item_profile.shape)
print(item_profile.head())
print('=======================================')

构建去重画像 (Profile Extraction)
(6040, 5)
     user_id  gender  age  occupation   zip
0          1       1    1          11  1589
53         2       2    7          17  2249
182       12       2    3          13  1166
205       15       2    3           8   905
406       17       2    6           2  3188
(3706, 1)
   movie_id
0      1105
1       640
2       854
3      3178
4      2163


In [ ]:

user_profile.set_index("user_id", inplace=True)
print('设置用户ID为索引 (Set User ID as Index)')
print(user_profile.head())
print('=======================================')


user_item_list = data.groupby("user_id")['movie_id'].apply(list)
print('构建用户-物品交互列表 (User-Item Interaction List)')
print(user_item_list.head())
print('=======================================')

设置用户ID为索引 (Set User ID as Index)
         gender  age  occupation   zip
user_id                               
1             1    1          11  1589
2             2    7          17  2249
12            2    3          13  1166
15            2    3           8   905
17            2    6           2  3188
构建用户-物品交互列表 (User-Item Interaction List)
user_id
1    [1105, 640, 854, 3178, 2163, 1108, 1196, 2600,...
2    [1105, 2890, 2129, 1783, 1118, 1849, 1155, 126...
3    [2163, 1108, 1179, 1782, 254, 2899, 628, 1121,...
4    [1026, 2489, 254, 1849, 3236, 1121, 467, 1107,...
5    [3178, 2163, 859, 2890, 1575, 2558, 145, 2489,...
Name: movie_id, dtype: object


In [6]:
#数据集切分#来自preprocess.py
train_set, test_set = gen_data_set(data, SEQ_LEN, negsample)  
print('数据集切分 (Dataset Splitting)')
print(len(train_set), len(test_set))
print(train_set[:5])
print(test_set[:5])
print('=======================================')

100%|██████████| 6040/6040 [00:04<00:00, 1319.26it/s]


8 8
数据集切分 (Dataset Splitting)
988129 6040
[(4890, 2496, 1, [51, 642, 1505, 650, 217, 1706, 1188, 1174, 3030, 2846, 2139, 81, 1156, 347, 310, 2893, 1575, 1557, 1638, 17, 34, 36, 1564, 266, 448, 2132, 594, 514, 2931, 2841, 2969, 2652, 1178, 1108, 2187, 2110, 1782], 37, [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 8, 8, 5, 8, 5, 8, 8, 8, 8, 4, 8, 8, 8, 8, 5, 6, 8, 5, 3, 8, 5, 5, 1, 5, 5, 8], 3, 5), (2063, 231, 1, [295, 363, 2243, 1531, 3035, 319, 1551, 75, 539, 2314, 2195, 737, 1823, 2235, 584, 2655, 128, 2312, 1304, 2255, 2491, 2148, 1477, 487, 3245, 936, 1483, 1708, 1375, 2116, 983, 1687, 2808, 2475, 2307, 2347, 1738, 413, 526, 320, 246, 2833, 1646, 1496, 22, 183, 2495, 394, 308, 799], 50, [1, 5, 8, 1, 5, 11, 1, 15, 1, 5, 5, 8, 1, 1, 5, 5, 16, 16, 3, 5, 5, 14, 1, 5, 5, 6, 5, 2, 1, 1, 1, 1, 1, 8, 5, 8, 1, 11, 5, 1, 5, 5, 5, 5, 6, 11, 1, 11, 1, 8], 5, 4), (1543, 1488, 1, [3465, 641, 160, 781, 574, 2297, 1051, 2097, 1585, 3513, 364, 1477, 2161, 3033, 1815, 2083, 1149, 461, 611, 864, 1538, 285, 2932, 

In [8]:
# 生成模型输入#来自preprocess.py
train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN)
test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN)
print('生成模型输入 (Generate Model Input)')

# 这是一个字典，不能直接用 train_model_input[:5] 切片
# 我们遍历字典，打印每个特征的前2条数据来看看
print("\n--- 训练集输入样例 (Train Input Sample) ---")
for key in ['user_id', 'hist_movie_id']: # 只看几个关键特征，避免刷屏
    if key in train_model_input:
        print(f"Feature: {key}")
        print(train_model_input[key][:2]) 

print("\n--- 训练集标签样例 (Train Label Sample) ---")
print(train_label[:5])

print("\n--- 测试集标签样例 (Test Label Sample) ---")
print(test_label[:5])
print('=======================================')

生成模型输入 (Generate Model Input)

--- 训练集输入样例 (Train Input Sample) ---
Feature: user_id
[4890 2063]
Feature: hist_movie_id
[[  51  642 1505  650  217 1706 1188 1174 3030 2846 2139   81 1156  347
   310 2893 1575 1557 1638   17   34   36 1564  266  448 2132  594  514
  2931 2841 2969 2652 1178 1108 2187 2110 1782    0    0    0    0    0
     0    0    0    0    0    0    0    0]
 [ 295  363 2243 1531 3035  319 1551   75  539 2314 2195  737 1823 2235
   584 2655  128 2312 1304 2255 2491 2148 1477  487 3245  936 1483 1708
  1375 2116  983 1687 2808 2475 2307 2347 1738  413  526  320  246 2833
  1646 1496   22  183 2495  394  308  799]]

--- 训练集标签样例 (Train Label Sample) ---
[1 1 1 1 1]

--- 测试集标签样例 (Test Label Sample) ---
[1 1 1 1 1]


In [ ]:
# 2.count #unique features for each sparse field and generate feature config for sequence feature

embedding_dim = 32

user_feature_columns = [SparseFeat('user_id', feature_max_idx['user_id'], 16),
                        SparseFeat("gender", feature_max_idx['gender'], 16),
                        SparseFeat("age", feature_max_idx['age'], 16),
                        SparseFeat("occupation", feature_max_idx['occupation'], 16),
                        SparseFeat("zip", feature_max_idx['zip'], 16),
                        VarLenSparseFeat(SparseFeat('hist_movie_id', feature_max_idx['movie_id'], embedding_dim,
                                                    embedding_name="movie_id"), SEQ_LEN, 'mean', 'hist_len'),
                        VarLenSparseFeat(SparseFeat('hist_genres', feature_max_idx['genres'], embedding_dim,
                                                   embedding_name="genres"), SEQ_LEN, 'mean', 'hist_len'),
                        ]

item_feature_columns = [SparseFeat('movie_id', feature_max_idx['movie_id'], embedding_dim)]

from collections import Counter
train_counter = Counter(train_model_input['movie_id'])
item_count = [train_counter.get(i,0) for i in range(item_feature_columns[0].vocabulary_size)]
sampler_config = NegativeSampler('frequency',num_sampled=255,item_name="movie_id",item_count=item_count)

In [ ]:
# 3.Define Model and train

import tensorflow as tf
if tf.__version__ >= '2.0.0':
    tf.compat.v1.disable_eager_execution()
else:
    K.set_learning_phase(True)
    
#model = YoutubeDNN(user_feature_columns, item_feature_columns, user_dnn_hidden_units=(128,64, embedding_dim), sampler_config=sampler_config)
model = ComiRec(user_feature_columns,item_feature_columns,k_max=2, user_dnn_hidden_units=(128,64, embedding_dim), sampler_config=sampler_config)

model.compile(optimizer="adam", loss=sampledsoftmaxloss)

history = model.fit(train_model_input, train_label,  # train_label,
                    batch_size=512, epochs=20, verbose=1, validation_split=0.0, )

# 4. Generate user features for testing and full item features for retrieval
test_user_model_input = test_model_input
all_item_model_input = {"movie_id": item_profile['movie_id'].values,}

user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)

user_embs = user_embedding_model.predict(test_user_model_input, batch_size=2 ** 12)
# user_embs = user_embs[:, i, :]  # i in [0,k_max) if MIND
item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12)

print(user_embs.shape)
print(item_embs.shape)

100%|██████████| 6040/6040 [00:11<00:00, 508.25it/s]


8 8
make sure the activation function use training flag properly call() got an unexpected keyword argument 'training'
make sure the activation function use training flag properly call() got an unexpected keyword argument 'training'
make sure the activation function use training flag properly call() got an unexpected keyword argument 'training'
make sure the activation function use training flag properly call() got an unexpected keyword argument 'training'
make sure the activation function use training flag properly call() got an unexpected keyword argument 'training'
make sure the activation function use training flag properly call() got an unexpected keyword argument 'training'
make sure the activation function use training flag properly call() got an unexpected keyword argument 'training'
make sure the activation function use training flag properly call() got an unexpected keyword argument 'training'
make sure the activation function use training flag properly call() got an unexpecte

# 使用faiss进行ANN查找并评估结果

In [4]:
! pip install faiss-cpu

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [4]:
import heapq
from collections import defaultdict
from tqdm import tqdm
import numpy as np
import faiss
from deepmatch.utils import recall_N

k_max = 2
topN = 50
test_true_label = {line[0]: [line[1]] for line in test_set}

index = faiss.IndexFlatIP(embedding_dim)
# faiss.normalize_L2(item_embs)
index.add(item_embs)
# faiss.normalize_L2(user_embs)

if len(user_embs.shape) == 2:  # multi interests model's shape = 3 (MIND,ComiRec)
    user_embs = np.expand_dims(user_embs, axis=1)

score_dict = defaultdict(dict)
for k in range(k_max):
    user_emb = user_embs[:, k, :]
    D, I = index.search(np.ascontiguousarray(user_emb), topN)
    for i, uid in tqdm(enumerate(test_user_model_input['user_id']), total=len(test_user_model_input['user_id'])):
        if np.abs(user_emb[i]).max() < 1e-8:
            continue
        for score, itemid in zip(D[i], I[i]):
            score_dict[uid][itemid] = max(score, score_dict[uid].get(itemid, float("-inf")))

s = []
hit = 0
for i, uid in enumerate(test_user_model_input['user_id']):
    pred = [item_profile['movie_id'].values[x[0]] for x in
            heapq.nlargest(topN, score_dict[uid].items(), key=lambda x: x[1])]
    filter_item = None
    recall_score = recall_N(test_true_label[uid], pred, N=topN)
    s.append(recall_score)
    if test_true_label[uid] in pred:
        hit += 1

print("recall", np.mean(s))
print("hr", hit / len(test_user_model_input['user_id']))

100%|██████████| 6040/6040 [00:01<00:00, 5487.52it/s]


recall 0.43642384105960264
hr 0.43642384105960264
